## Función genérica que exporta los datos en CSV para posterior análisis

Genera un CSV por llamada

In [ ]:
import pandas as pd
import numpy as np
import csv

def save_sampled_data(data, algorithm_labels: list, file_prefix: str, sample_rate: int = 20, is_arm_statistics: bool = False):
    """
    Guarda una matriz de datos en un archivo CSV con muestreo de cada `sample_rate` pasos.
    También permite guardar estadísticas de brazos si `is_arm_statistics` es True.

    :param data: Matriz con los datos a guardar (puede ser rewards, optimal selections, regret, etc.)
                 o lista de estadísticas de brazos si is_arm_statistics=True.
    :param algorithm_labels: Lista con los nombres de los algoritmos correspondientes a las columnas.
    :param file_prefix: Prefijo del nombre del archivo (por ejemplo, "epsilon_greedy_rewards").
    :param sample_rate: Frecuencia de muestreo en los pasos (por defecto, cada 20 pasos).
    :param is_arm_statistics: Indica si los datos corresponden a estadísticas de brazos (True) o a matrices (False).
    """

    file_name = f"data/{file_prefix}_sampled.csv"

    if is_arm_statistics:
        # Guardar estadísticas de brazos
        with open(file_name, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(['Algorithm', 'Arm', 'Mean_Reward', 'Count'])
            for idx, algo in enumerate(data):  # `data` aquí es `arm_stats`
                for arm_index, (mean, count) in enumerate(zip(algo["means"], algo["counts"])):
                    writer.writerow([algorithm_labels[idx], arm_index, mean, count])
    
    else:
        # Guardar datos estructurados (Recompensa, Selecciones Óptimas, Regret)
        steps_sampled = np.arange(0, data.shape[1], sample_rate)
        df_sampled = pd.DataFrame(data[:, ::sample_rate].T, columns=algorithm_labels, index=steps_sampled)
        df_sampled.to_csv(file_name, index_label="Step", float_format="%.2f")

    print(f"Archivo guardado: {file_name}")




## Epsilon-Greedy

Llamadas a la función para generar los CSV representativos de las gráficas

In [ ]:

# Construir etiquetas de los algoritmos basadas en su epsilon
algorithm_labels = [f"EpsilonGreedy (epsilon={algo.epsilon})" for algo in algorithms]

# Guardar los datos de recompensa promedio
save_sampled_data(rewards, algorithm_labels, "epsilon_greedy_rewards")

# Guardar los datos de porcentaje de selecciones del brazo óptimo
save_sampled_data(optimal_selections, algorithm_labels, "epsilon_greedy_optimal_selections")

# Guardar los datos de regret acumulado
save_sampled_data(regret_accumulated, algorithm_labels, "epsilon_greedy_regret")

# Guardar estadísticas de brazos
save_sampled_data(arm_stats, algorithm_labels, "epsilon_greedy_arm_statistics", is_arm_statistics=True)

